In [1]:
import pandas as pd
import pickle
from utils import model as mod

In [2]:
df = pd.read_pickle('../data/03_df_data')
segs = pd.read_pickle('../data/02_df_seg_race')

with open("../data/02_dic_ref_groups.pkl", "rb") as file:
    ref_groups = pickle.load(file)

In [3]:
df.head()

,loan_type,loan_purpose,preapproval,construction_method,occupancy_type,loan_amount,action_taken,state_code,county_code,census_tract,...,interest_only_payment,property_value,manufactured_home_secured_property_type,manufactured_home_land_property_interest,total_units,open_end_line_of_credit,business_or_commercial_purpose,reg_uw,denied,reg_price
1,Conventional,Home purchase,Not requested,Site built,Primary residence,1205000,Purchased,WA,53033.0,53033032318.0,...,Not interest only,1505000.0,Not applicable,Not applicable,1,Not open end credit,Not business commercial,0,0,0
2,Conventional,Home purchase,Not requested,Site built,Primary residence,925000,Purchased,WA,53011.0,53011040303.0,...,Not interest only,1035000.0,Not applicable,Not applicable,1,Not open end credit,Not business commercial,0,0,0
5,Conventional,Home purchase,Not requested,Site built,Primary residence,905000,Purchased,WA,53011.0,53011040905.0,...,Not interest only,1135000.0,Not applicable,Not applicable,1,Not open end credit,Not business commercial,0,0,0
13,Conventional,Home purchase,Not requested,Site built,Second residence,1505000,Purchased,CA,6065.0,6065045606.0,...,Not interest only,4295000.0,Not applicable,Not applicable,1,Not open end credit,Not business commercial,0,0,0
14,Conventional,Home purchase,Not requested,Site built,Investment,605000,Purchased,CA,6059.0,6059062641.0,...,Not interest only,1645000.0,Not applicable,Not applicable,1,Not open end credit,Business commercial,0,0,0


In [4]:
segs

,loan_type,loan_purpose,applicant_race_1,applied,event_count,event_rate,mi_rate,mu_rate,mx_rate,crit event,crit count
1,Conventional,Cash out refinance,Black African American,1761,257,0.145940,4.490,6.737013,9.740,True,True
2,Conventional,Cash out refinance,White,9223,1274,0.138133,2.875,6.765602,10.240,True,True
4,Conventional,Home improvement,Black African American,541,128,0.236599,4.750,7.659269,10.240,True,True
5,Conventional,Home improvement,White,3644,669,0.183589,4.625,7.822651,10.240,True,True
6,Conventional,Home purchase,Asian,6913,458,0.066252,2.625,6.136274,8.990,True,True
7,Conventional,Home purchase,Black African American,4818,502,0.104193,2.500,6.380016,9.240,True,True
8,Conventional,Home purchase,White,36101,2336,0.064707,1.000,6.375974,9.490,True,True
9,Conventional,Refinance,Asian,1701,137,0.080541,3.750,5.902198,8.030,True,True
11,Conventional,Refinance,White,7497,559,0.074563,4.250,6.179631,8.375,True,True
18,FHA insured,Home purchase,Black African American,884,125,0.141403,4.125,6.339181,7.750,True,True


In [5]:
testing = segs.columns.tolist()[2]

In [6]:
ref_groups

{'applicant_race_1': 'White',
 'applicant_sex': 'Male',
 'applicant_age_above_62': 'No'}

In [7]:
segs.loan_type.unique().tolist()

['Conventional', 'FHA insured']

In [8]:
drop_this = [
    'applicant_ethnicity_1',        
    'applicant_race_1',                  
    'applicant_sex', 
    'applicant_age_above_62'
]

In [11]:
# %%capture output

for type_i in segs.loan_type.unique().tolist():
    print(f'\nType: {type_i}')
    
    for purp_i in segs[segs['loan_type'] == type_i].loan_purpose.unique().tolist():
        print(f'\n Purpose: {purp_i}')

        group_list = segs[(segs['loan_type'] == type_i)&(segs['loan_purpose'] == purp_i)][testing].unique().tolist()
        group_list.remove(ref_groups[testing])
        for group_i in group_list:
            print(f'\n  PB: {group_i}')
            print(f'  Ref: {ref_groups[testing]}')
    
            """
            Underwriting
            """

            print('\nUnderwriting')
    
            df_tmp = df[
                (df.loan_type == type_i) &
                (df.loan_purpose == purp_i) &
                (df[testing].isin([ref_groups[testing]] +  [group_i])) &
                (df['reg_uw'] == 1)
            ]



            
            
            print(f'shape: {df_tmp.shape}')

            print(df_tmp.groupby(testing).agg(
                count=('denied','size'),
                event_count=('denied','sum')
            ))

            # make dummy for protected basis group

            df_tmp[group_i] = (df_tmp[testing] == group_i).astype(int)

            df_tmp.drop(columns = drop_this, inplace = True)
            
            # drop columns with single value. will end up being things like indicator for segment
            df_tmp = df_tmp.loc[:, df_tmp.nunique(dropna=False) > 1]
            

            print('')
            print(df_tmp[group_i].value_counts())
            
            """
            UW Model 0
            """

            print('\nModel 0')
            
            import statsmodels.api as sm
            
            X = sm.add_constant(df_tmp[group_i])
            y = df_tmp["denied"]
            
            
            model = sm.Logit(y, X)
            result = model.fit()
            
            print(result.summary())
    
    
            
            """
            UW Model 1
            """
            print('\nModel 1')

            result = mod.logistic_woe_run(df_tmp,group_i)
            
            
            """
            UW Model 2
            """
            print('\nModel 2')

            df_tmp_psa = mod.make_match_pair(df_tmp,'denied',group_i)
            
            result = mod.logistic_woe_run(df_tmp_psa,group_i)











            
            
    
            """
            Pricing
            """
            print('\nPricing')
                
            df_tmp = df[
                (df.loan_type == type_i) &
                (df.loan_purpose == purp_i) &
                (df[testing].isin([ref_groups[testing]] +  [group_i])) &
                (df['reg_price'] == 1)
            ]



            
            
            print(f'shape: {df_tmp.shape}')

            # make dummy for protected basis group

            df_tmp[group_i] = (df_tmp[testing] == group_i).astype(int)

            df_tmp.drop(columns = drop_this, inplace = True)
            
            # drop columns with single value. will end up being things like indicator for segment
            df_tmp = df_tmp.loc[:, df_tmp.nunique(dropna=False) > 1]
            

            print('')
            print(df_tmp[group_i].value_counts())
    
            
            """
            Pri Model 0
            """
            print('\nModel 0')
            
            import statsmodels.api as sm
            
            X = sm.add_constant(df_tmp[group_i])
            y = df_tmp["interest_rate"]
            
            
            model = sm.OLS(y, X)
            result = model.fit()
            
            print(result.summary())
    
    
    
    
            
            """
            Pri Model 1
            """
            print('\nModel 1')

            result = mod.ols_dummy_run(df_tmp,group_i)

            print(result.summary())

            
    
            """
            Pri Model 2
            """

            print('\nModel 2')

            #pair function needs to ignore something, gets excluded later bc singular value
            df_tmp['rnd'] = 999
            df_tmp['action_taken'] = 'junk'
            
            df_tmp_psa = mod.make_match_pair(df_tmp,'rnd',group_i)
            result = mod.ols_dummy_run(df_tmp_psa,group_i)

            print(result.summary())

            
            
            break # group_i

        break # purp_i
    break # type_i




        

In [ ]:
print(
    'hello'
)